In [2]:
!pip install -q transformers datasets pillow accelerate huggingface_hub

import torch
from datasets import load_dataset
from torch.utils.data import DataLoader
from transformers import BlipProcessor, BlipForConditionalGeneration

device = "cuda" if torch.cuda.is_available() else "cpu"

# -------------------------
# Load RICO dataset
# -------------------------
dataset = load_dataset("rootsautomation/RICO-Screen2Words")["train"]

dataset = dataset.shuffle(seed=42)

# small subset for Kaggle training
train_dataset = dataset.select(range(800))


# -------------------------
# Preprocess dataset
# -------------------------
def preprocess(example):
    return {
        "image": example["image"],
        "caption": example["captions"][0]
    }

train_dataset = train_dataset.map(
    preprocess,
    remove_columns=train_dataset.column_names,
    batched=False
)

print("Dataset size:", len(train_dataset))

README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00008.parquet:   0%|          | 0.00/216M [00:00<?, ?B/s]

data/train-00001-of-00008.parquet:   0%|          | 0.00/219M [00:00<?, ?B/s]

data/train-00002-of-00008.parquet:   0%|          | 0.00/225M [00:00<?, ?B/s]

data/train-00003-of-00008.parquet:   0%|          | 0.00/215M [00:00<?, ?B/s]

...

Generating train split:   0%|          | 0/15743 [00:00<?, ? examples/s]

Generating val split:   0%|          | 0/2364 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/4310 [00:00<?, ? examples/s]

Map:   0%|          | 0/800 [00:00<?, ? examples/s]

Dataset size: 800


In [3]:
model_name = "Salesforce/blip-image-captioning-base"

processor = BlipProcessor.from_pretrained(model_name)
model = BlipForConditionalGeneration.from_pretrained(model_name)

model.to(device)


# -------------------------
# Collate function
# -------------------------
def collate_fn(batch):

    images = [item["image"] for item in batch]
    captions = [item["caption"] for item in batch]

    inputs = processor(
        images=images,
        text=captions,
        padding=True,
        return_tensors="pt"
    )

    inputs["labels"] = inputs["input_ids"]

    return {k: v.to(device) for k, v in inputs.items()}


# -------------------------
# DataLoader
# -------------------------
train_loader = DataLoader(
    train_dataset,
    batch_size=4,   # safe for T4
    shuffle=True,
    collate_fn=collate_fn
)

preprocessor_config.json:   0%|          | 0.00/287 [00:00<?, ?B/s]

The image processor of type `BlipImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/506 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/990M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie text_decoder.cls.predictions.bias to text_decoder.cls.predictions.decoder.bias, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie text_decoder.bert.embeddings.word_embeddings.weight to text_decoder.cls.predictions.decoder.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
BlipForConditionalGeneration LOAD REPORT from: Salesforce/blip-image-captioning-base
Key                                       | Status     |  | 
------------------------------------------+------------+--+-
text_decoder.bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identic

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

In [4]:
from torch.optim import AdamW
from tqdm import tqdm

optimizer = AdamW(model.parameters(), lr=2e-5)

model.train()

for epoch in range(1):

    total_loss = 0

    progress_bar = tqdm(train_loader)

    for batch in progress_bar:

        outputs = model(**batch)

        loss = outputs.loss

        loss.backward()

        optimizer.step()
        optimizer.zero_grad()

        total_loss += loss.item()

        progress_bar.set_description(f"Loss: {loss.item():.4f}")

    print("Epoch loss:", total_loss)


# -------------------------
# Save model locally
# -------------------------
model.save_pretrained("rico_blip_model")
processor.save_pretrained("rico_blip_model")


# -------------------------
# Example inference
# -------------------------
model.eval()

sample = train_dataset[0]

with torch.no_grad():

    inputs = processor(
        images=sample["image"],
        return_tensors="pt"
    ).to(device)

    output = model.generate(**inputs, max_new_tokens=30)

caption = processor.decode(output[0], skip_special_tokens=True)

print("Generated caption:", caption)


Loss: 3.1885: 100%|██████████| 200/200 [02:14<00:00,  1.48it/s]

Epoch loss: 721.8444801568985


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Generated caption: display of a page showing of a fitness app


In [7]:
from huggingface_hub import notebook_login

notebook_login()

model.push_to_hub("rico-blip-ui-caption")
processor.push_to_hub("rico-blip-ui-caption")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

README.md: 0.00B [00:00, ?B/s]

CommitInfo(commit_url='https://huggingface.co/preksham2004/rico-blip-ui-caption/commit/4809ae764441abee1add2e5cbdd3d059839dd0e6', commit_message='Upload processor', commit_description='', oid='4809ae764441abee1add2e5cbdd3d059839dd0e6', pr_url=None, repo_url=RepoUrl('https://huggingface.co/preksham2004/rico-blip-ui-caption', endpoint='https://huggingface.co', repo_type='model', repo_id='preksham2004/rico-blip-ui-caption'), pr_revision=None, pr_num=None)